In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import re

RUN_ID = "20260810_104157_31209"
CSV_DIR = Path("..") / "results" / RUN_ID / "csv"

trace_df = pd.read_csv(CSV_DIR / "layerwise_hessian_traces.csv")
eig_df = pd.read_csv(CSV_DIR / "layerwise_top_eigenvalues.csv")
class_df = pd.read_csv(CSV_DIR / "classification_metrics.csv")
error_df = pd.read_csv(CSV_DIR / "layerwise_quant_error.csv")

PLOT_DIR = Path("plots") / RUN_ID
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# sanity check: every combo should have 3 stages
print(trace_df.groupby(["model", "dataset"])["stage"].nunique().value_counts())

def abbreviate(name):
    mapping = {"layer": "l", "conv": "c", "weight": "w", "downsample" : "d"}
    result = []
    for idx in name.split('.'):
        sub_name = re.findall(r'[a-zA-Z]+|\d+', idx)
        abb_sub = []
        for sn in sub_name:
            abb_sub.append(mapping.get(sn, sn))
        result.append("".join(abb_sub))
    return '.'.join(result)

stage
3    3
Name: count, dtype: int64


In [2]:
def plot_layerwise_metric(df, model, dataset, metric, save=False):
    # filter to this model+dataset
    subset = df[(df["model"] == model) & (df["dataset"] == dataset)]

    # establish a single consistent layer order from the FP32 rows
    layer_order = subset[subset["stage"] == "FP32"]["layer"].tolist()

    fig, ax = plt.subplots(figsize=(14, 6))

    # plot one line per stage, x = layer index, y = metric value
    for stage in ["FP32", "PTQ", "QAT"]:
        stage_df = subset[subset["stage"] == stage]
        stage_values = (
            stage_df.set_index("layer")[metric].reindex(layer_order)
        )
        x = range(len(layer_order))
        ax.plot(x, stage_values.values, marker="o", label=stage)

    ax.set_yscale("log")
    
    ax.set_xticks(range(len(layer_order))) 
    ax.set_xticklabels([abbreviate(name) for name in layer_order], rotation=90)

    metric_label = "Hessian Trace" if metric == "trace" else "Top Eigenvalue"
    ax.set_xlabel("Layer (input \u2192 output)")
    ax.set_ylabel(metric_label)
    ax.set_title(f"{model} on {dataset} \u2014 {metric_label}")
    ax.legend()
    ax.grid(True, which="both", axis="y", alpha=0.3)
    
    fig.tight_layout()
    if save:
        fname = PLOT_DIR / f"{metric}_{model}_{dataset}.png"
        fig.savefig(fname, dpi=150)
    
    return fig, ax

In [3]:
sources = [(trace_df, "trace"), (eig_df, "eigenvalue")]

for df, metric in sources:
    combos = df[["model", "dataset"]].drop_duplicates()
    for _, row in combos.iterrows():
        model = row["model"]
        dataset = row["dataset"]
        fix , ax = plot_layerwise_metric(trace_df, model, dataset, "trace", save=True)
        plt.close()
        
for df, metric in sources:
    combos = df[["model", "dataset"]].drop_duplicates()
    for _, row in combos.iterrows():
        model = row["model"]
        dataset = row["dataset"]
        fix , ax = plot_layerwise_metric(eig_df, model, dataset, "eigenvalue", save=True)
        plt.close()

In [4]:
run_pattern = re.compile(r"--- Run \d+/\d+: (?P<model>\S+) on (?P<dataset>\S+) ---")
metric_pattern = re.compile(r"\[(?P<stage>FP32|PTQ|QAT)\].*Val Acc: (?P<acc>[\d.]+)% \| Latency: (?P<latency>[\d.]+)ms")

def parse_log(log_path):
    records = []
    current_model, current_dataset = None, None

    with open(log_path) as f:
        for line in f:
            run = run_pattern.search(line)
            if run:
                current_model = run.group("model")
                current_dataset = run.group("dataset")
            metric = metric_pattern.search(line)
            if metric:
                stage = metric.group("stage")
                acc = metric.group("acc")
                latency = metric.group("latency")
                
                record = {
                    "model": current_model,
                    "dataset": current_dataset,
                    "stage": stage,
                    "acc": float(acc),
                    "latency": float(latency),
                }
                records.append(record)
            pass

    return pd.DataFrame(records)

In [5]:
def plot_acc_vs_latency(df, model, dataset, save=False):
    # filter to this model+dataset
    
    subset = df[(df["model"] == model) & (df["dataset"] == dataset)]

    # establish a single consistent layer order from the FP32 rows
    model_type = ["FP32" , "PTQ" , "QAT"]
    
    stage = subset.set_index("stage")
    acc_values = stage["acc"].reindex(model_type)
    latency_values = stage["latency"].reindex(model_type)
    
    # x = range(len(model_type))

    fig, ax = plt.subplots(figsize=(8, 5))
    # ax2 = ax.twinx()
    
    line, = ax.plot(latency_values.values, acc_values.values, marker  = "o", color="tab:blue")
    # line2, = ax.plot(x, latency_values.values, marker= "s", color="tab:red", label="Latency")
    for models in model_type:
        x_val = latency_values[models]
        y_val = acc_values[models]
        ax.annotate(models, (x_val, y_val), textcoords="offset points", xytext=(5,5))
    # ax.set_xticks(list(x))
    # ax.set_xticklabels(model_type)
    ax.set_xlabel("Latency (ms)")
    ax.set_ylabel("Accuracy (%)")
    # ax2.set_ylabel("Latency (ms)", color="tab:red")
    # ax.set_xscale("log")
    ax.tick_params(axis="y", labelcolor="tab:blue")
    # ax2.tick_params(axis="y", labelcolor="tab:red")
    
    ax.set_title(f"{model} on {dataset}")
    # ax.legend(handles=[line], loc="best")
    
    ax.grid(True, axis="y", alpha=0.3)
    fig.tight_layout()
    
    if save:
        fname = PLOT_DIR / f"acc_vs_latency_{model}_{dataset}.png"
        fig.savefig(fname, dpi=150)
    
    return fig, ax

In [7]:
LOG_DIR = Path("..") / "results" / "logs"
LOG_PATH = LOG_DIR / "job_31209.out"

acc_lat_df = parse_log(LOG_PATH)
combos = acc_lat_df[["model", "dataset"]].drop_duplicates()
for _, row in combos.iterrows():
    fig, ax = plot_acc_vs_latency(acc_lat_df, row["model"], row["dataset"], save=True)
    plt.close(fig)

In [8]:
def plot_layerwise_error(df, model, dataset, metric, save=False):
    # filter to this model and dataset combination
    subset = df[(df["model"] == model) & (df["dataset"] == dataset)]

    # layer order taken from PTQ rows, no FP32 stage present in this data
    layer_order = subset[subset["stage"] == "PTQ"]["layer"].tolist()

    fig, ax = plt.subplots(figsize=(14, 6))
    colors = {"PTQ": "tab:blue", "QAT": "tab:orange"}
    x = range(len(layer_order))

    for stage in ["PTQ", "QAT"]:
        stage_df = subset[subset["stage"] == stage]
        values = stage_df.set_index("layer")[metric].reindex(layer_order)

        if metric == "sqnr":
            # inf occurs when mse is exactly 0, drop it to keep axis readable
            values = values.replace([float("inf"), float("-inf")], float("nan"))

        ax.plot(x, values.values, marker="o", color=colors[stage], label=stage)

    # if metric == "mse":
    #     ax.set_yscale("log")

    ax.set_xticks(list(x))
    ax.set_xticklabels([abbreviate(name) for name in layer_order], rotation=90)
    ax.set_xlabel("Layer (input -> output)")

    metric_label = {"mse": "MSE (log scale)", "sqnr": "SQNR (dB)"}.get(metric, metric)
    ax.set_ylabel(metric_label)
    ax.set_title(f"{model} on {dataset} - layerwise {metric_label}")

    ax.legend()
    ax.grid(True, which="both", axis="y", alpha=0.3)
    fig.tight_layout()

    if save:
        fig.savefig(PLOT_DIR / f"{model}_{dataset}_{metric}.png", dpi=150)

    return fig, ax

In [9]:
combos = error_df[["model", "dataset"]].drop_duplicates()

for _, row in combos.iterrows():
    model = row["model"]
    dataset = row["dataset"]
    fig1 , ax1 = plot_layerwise_error(error_df, model, dataset, "mse",  save=True)
    plt.close(fig1)
    
    fig2 , ax2 = plot_layerwise_error(error_df, model, dataset, "sqnr",  save=True)
    plt.close(fig2)